<a href="https://colab.research.google.com/github/Joanachoong/Pfizer-Advanced-AI-Powered-Document-Insights-Data-Extraction-Externship/blob/main/Pfizer_Build_the_Full_RAG_UI_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Document Processing — Handle both digital PDFs and scanned documents.
Apply OCR for scanned files when needed. Extract clean text and chunk it properly.

Metadata Tagging — Tag chunks with document type, page ranges, and source identifiers. Use consistent metadata for filtering and retrieval.

Embeddings & Indexing — Use an open-source embedding model (e.g., sentence-transformers). Store and retrieve chunks using FAISS or LlamaIndex.

Smart Prompting — Build clear, context-grounded prompts. Instruct the model to cite sources in its answers

Open-Source Model — Use an open-source LLM (Hugging Face, Ollama, etc.) — not Gemini. Ensure the model integrates smoothly with your retrieval logic.

Complete RAG Pipeline — Retrieval → Context Building → Prompt → Model → Answer + Sources. Include confidence scores and chunk count in responses.

User Interface — Build a Gradio UI that allows users to upload documents, view chat history, and see answers with sources and confidence levels. Keep it clean and intuitive — think product demo, not debug tool.

# Part 0 : Install all required libraries ( this will take a while )


In [ ]:
!pip install PyMuPDF

!pip install -q pymupdf paddleocr pdf2image pillow numpy
!pip install -q llama-index llama-index-embeddings-huggingface sentence-transformers
!apt-get install -y poppler-utils -q  # required by pdf2image

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.13).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.


In [ ]:
# ============================================
# Import Libraries
# ============================================
import re
import unicodedata
import numpy as np

import fitz  # PyMuPDF
from paddleocr import PaddleOCR
from pdf2image import convert_from_path

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import Document
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [ ]:
from google.colab import files

print("Upload the PDF you want to run through the pipeline:")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\nUploaded: {pdf_path}")

Upload the PDF you want to run through the pipeline:


Saving Sample Documentation.pdf to Sample Documentation.pdf

Uploaded: Sample Documentation.pdf


# Part 1 : Document Processing

## Process Workflow

                 Input PDF
                     │
                     ▼
          ┌──────────────────────┐
          │ Extract Text (PyMuPDF)│
          └──────────────────────┘
                     │
                     ▼
          Is extracted text usable?
               /              \
             Yes              No
              │                │
              ▼                ▼
      Clean & Normalize   Convert PDF to Images
              │                │
              │                ▼
              │         PaddleOCR Processing
              │                │
              └──────────┬─────┘
                         ▼
                 Clean OCR Output
                         ▼
                 Text Chunking
                         ▼
              Embedding Generation
                         ▼
                 Vector Database



In [ ]:
# ============================================
# extract_pdf_text()
# ============================================
def extract_pdf_text(pdf_path):
    """
    Extract machine-readable text from a PDF using PyMuPDF.

    Returns
    -------
    str
        Concatenated text from all pages.
    """
    text_parts = []
    doc = fitz.open(pdf_path)
    for page in doc:
        text_parts.append(page.get_text())
    doc.close()
    return "\n".join(text_parts)

In [ ]:
# ============================================
# is_text_usable()
# ============================================
def is_text_usable(text, min_chars=50, min_words=10, min_printable_ratio=0.85):
    """
    Determine whether extracted text is usable.

    Returns True if:
    - sufficient characters
    - sufficient words
    - reasonable printable ratio
    """
    if text is None:
        return False

    stripped = text.strip()
    if len(stripped) < min_chars:
        return False

    word_count = len(stripped.split())
    if word_count < min_words:
        return False

    printable_count = sum(1 for ch in stripped if ch.isprintable())
    printable_ratio = printable_count / len(stripped) if stripped else 0
    if printable_ratio < min_printable_ratio:
        return False

    return True

In [ ]:

!pip install easyocr

import easyocr

print("\nEasyOCR installed and imported!")




EasyOCR installed and imported!


In [ ]:
def process_document(pdf_path):
    """
    PyMuPDF first -> check usable -> EasyOCR fallback (on rasterized pages).
    """
    text = extract_pdf_text(pdf_path)

    if is_text_usable(text):
        return text

    # Fallback: rasterize each page to an image, then OCR the images
    import fitz
    reader = easyocr.Reader(['en'])
    doc = fitz.open(pdf_path)
    ocr_text_parts = []
    for page in doc:
        pix = page.get_pixmap(dpi=200)
        img_bytes = pix.tobytes("png")
        results = reader.readtext(img_bytes, detail=0)
        ocr_text_parts.append("\n".join(results))
    doc.close()
    return "\n".join(ocr_text_parts)

In [ ]:
import easyocr
import os
# ---- TEST: run extraction end-to-end and sanity-check the output ----
print(pdf_path)
print(os.path.exists(pdf_path))
raw_text = process_document(pdf_path)

print(f"Extracted {len(raw_text)} characters, {len(raw_text.split())} words")
print("-" * 60)
print(raw_text[:500])
print("-" * 60)
assert is_text_usable(raw_text), "Extraction produced unusable text — inspect the PDF/OCR output above."
print("Block 1 passed: raw_text is populated and usable.")

Sample Documentation.pdf
True


Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% CompleteExtracted 62733 characters, 9242 words
------------------------------------------------------------
0
BASF
We create
chemistry
RegXcellencef
Kollidone SR
PRD-No:: 30071321
Product Specification
Workflow No.
DAWF-2024-0299
Revision 8
Effective from: 01 Jul 2024
Status: final
Test parameter
Requirements
Test method
Appearance
white to slightly yellowish,
visual
free flowing powder
Identification
Identification (IR)
Must comply
PM/00511
Identification (sulfate)
Must comply
Ph.Eur. ("sodium lauril
sulfate" , ID test C)
Purity tests
Content of Povidone
18.0 to 21.0 g/100g
%
PM/02047
(calculated: Ni
------------------------------------------------------------
Block 1 passed: raw_text is populated and usable.


In [ ]:
# ============================================
# clean_text()
# ============================================
def clean_text(text):
    """
    Responsibilities:
    - normalize spaces
    - remove duplicated blank lines
    - fix OCR spacing
    - normalize Unicode
    - remove page artefacts
    """
    # normalize Unicode (e.g. curly quotes, ligatures -> standard forms)
    text = unicodedata.normalize("NFKC", text)

    # remove common page artefacts: 'Page 3 of 10', standalone page numbers, form-feed chars
    text = re.sub(r"Page\s+\d+\s+of\s+\d+", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"^\s*\d+\s*$", " ", text, flags=re.MULTILINE)
    text = text.replace("\x0c", "\n")

    # fix OCR spacing: stray spaces before punctuation, hyphenated line-breaks
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)          # de-hyphenate wrapped words
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)           # 'word .' -> 'word.'

    # normalize whitespace: collapse repeated spaces/tabs
    text = re.sub(r"[ \t]+", " ", text)

    # remove duplicated blank lines (3+ newlines -> 2)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
# ---- TEST: show a before/after diff on the raw extraction ----
cleaned_text = clean_text(raw_text)

print("BEFORE (first 300 chars):")
print(repr(raw_text[:500]))
print()
print("AFTER (first 300 chars):")
print(repr(cleaned_text[:500]))
print()
print(f"Length before: {len(raw_text)} | Length after: {len(cleaned_text)}")

BEFORE (first 300 chars):
'0\nBASF\nWe create\nchemistry\nRegXcellencef\nKollidone SR\nPRD-No:: 30071321\nProduct Specification\nWorkflow No.\nDAWF-2024-0299\nRevision 8\nEffective from: 01 Jul 2024\nStatus: final\nTest parameter\nRequirements\nTest method\nAppearance\nwhite to slightly yellowish,\nvisual\nfree flowing powder\nIdentification\nIdentification (IR)\nMust comply\nPM/00511\nIdentification (sulfate)\nMust comply\nPh.Eur. ("sodium lauril\nsulfate" , ID test C)\nPurity tests\nContent of Povidone\n18.0 to 21.0 g/100g\n%\nPM/02047\n(calculated: Ni'

AFTER (first 300 chars):
'BASF\nWe create\nchemistry\nRegXcellencef\nKollidone SR\nPRD-No:: 30071321\nProduct Specification\nWorkflow No.\nDAWF-2024-0299\nRevision 8\nEffective from: 01 Jul 2024\nStatus: final\nTest parameter\nRequirements\nTest method\nAppearance\nwhite to slightly yellowish,\nvisual\nfree flowing powder\nIdentification\nIdentification (IR)\nMust comply\nPM/00511\nIdentification (sulfate)\nMust comply\nPh.Eur. ("sod

In [ ]:
"""
Step 1: Install required libraries
"""

# Install PyTorch (CUDA support)
!pip install -q torch

# Check CUDA version, then install llama-cpp-python with matching CUDA build
!nvcc --version
!pip install --no-cache-dir llama-cpp-python==0.2.90 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu123

# LlamaIndex core + integrations needed for Mistral + embeddings
!pip install llama-index
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-llama-cpp --no-deps


!pip install -q llama-index-core

# Install PyMuPDF (fitz) for PDF text extraction and pandas for tabular output
!pip install -q PyMuPDF pandas

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu123


In [ ]:
"""
Step 2. Download and verify the Mistral GGUF file
"""
from llama_cpp import Llama
import os

model_path = "/content/mistral.gguf"

# Remove any previously corrupted/incomplete download
if os.path.exists(model_path):
    os.remove(model_path)

# Download Mistral model if not already present
if not os.path.exists(model_path):
    !wget -c https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print(f"Model downloaded to {model_path}")

# Verify file exists and check size
if os.path.exists(model_path):
    print(f"Model file exists. Size: {os.path.getsize(model_path) / (1024 * 1024):.2f} MB")
else:
    print("Model file not found!")

# Load the model with GPU acceleration
try:
    llm_check = Llama(
        model_path=model_path,
        n_gpu_layers=1,   # Start with 1 layer on GPU to be safe
        n_ctx=2048,       # Context window size
        verbose=True      # Show loading progress
    )
    print("Model loaded successfully!")
except Exception as e:
    print(f"Error loading model: {e}")

--2026-08-31 16:24:28--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.165.160.12, 3.165.160.11, 3.165.160.59, ...
Connecting to huggingface.co (huggingface.co)|3.165.160.12|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65778ac662d3ac1817cc9201/865f5e4682dddb29c2e20270b2471a7590c83a414bbf1d72cf4c08fdff2eeca4?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.2.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.2.Q4_K_M.gguf%22%3B&Expires=1788197068&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjU3NzhhYzY2MmQzYWMxODE3Y2M5MjAxLzg2NWY1ZTQ2ODJkZGRiMjljMmUyMDI3MGIyNDcxYTc1OTBjODNhNDE0YmJmMWQ3MmNmNGMwOGZkZmYyZWVjYTRcXD91c2VyX2lkPXB1YmxpYyZYLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRp

llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

Model downloaded to /content/mistral.gguf
Model file exists. Size: 4166.07 MB


llm_load_tensors:        CPU buffer size =  4165.37 MiB
.................................................................................................
llama_new_context_with_model: n_ctx      = 2048
llama_new_context_with_model: n_batch    = 512
llama_new_context_with_model: n_ubatch   = 512
llama_new_context_with_model: flash_attn = 0
llama_new_context_with_model: freq_base  = 1000000.0
llama_new_context_with_model: freq_scale = 1
llama_kv_cache_init:        CPU KV buffer size =   256.00 MiB
llama_new_context_with_model: KV self size  =  256.00 MiB, K (f16):  128.00 MiB, V (f16):  128.00 MiB
llama_new_context_with_model:        CPU  output buffer size =     0.12 MiB
llama_new_context_with_model:        CPU compute buffer size =   164.01 MiB
llama_new_context_with_model: graph nodes  = 1030
llama_new_context_with_model: graph splits = 1


Model loaded successfully!


AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | AVX512_BF16 = 0 | FMA = 1 | NEON = 0 | SVE = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | MATMUL_INT8 = 0 | LLAMAFILE = 1 | 
Model metadata: {'tokenizer.chat_template': "{{ bos_token }}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}{{ raise_exception('Conversation roles must alternate user/assistant/user/assistant/...') }}{% endif %}{% if message['role'] == 'user' %}{{ '[INST] ' + message['content'] + ' [/INST]' }}{% elif message['role'] == 'assistant' %}{{ message['content'] + eos_token}}{% else %}{{ raise_exception('Only user and assistant roles are supported!') }}{% endif %}{% endfor %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.padding_token_id': '0', 'tokenizer.ggml.unknown_token_id': '0', 'tokenizer.ggml.eos_token_id': '2', 'general.architecture': 'llama', 'llama.rope.freq_base': 

In [ ]:
"""
Wrap Mistral in LlamaIndex's LlamaCPP interface and expose run_mistral()
for reuse everywhere else in the notebook (Part 2's classifier calls this;
Part 5 builds the retrieval-based answer_question() on top of it later).
"""
from llama_index.core.settings import Settings
from llama_index.llms.llama_cpp import LlamaCPP

llm = LlamaCPP(
    model_path="/content/mistral.gguf",
    temperature=0.7,
    max_new_tokens=512,
    context_window=2048,
    model_kwargs={"n_gpu_layers": 1}
)

Settings.llm = llm

def run_mistral(prompt, max_tokens=512):
    """Thin wrapper so any code -- Part 2's classifier included -- can call Mistral."""
    response = llm.complete(prompt, max_tokens=max_tokens)
    return str(response)


llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /content/mistral.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.

# Part 2: Metadata Tagging


- perform text tagging for more fast and accurate retrival
- perform chuncking

In [ ]:
# ============================================================
# STEP 1: Install Required Libraries
# ============================================================
# Install llama-cpp-python pre-built wheel with CUDA support for fast GPU inference
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122

# Install PyMuPDF (fitz) for PDF text extraction and pandas for tabular output
!pip install -q PyMuPDF pandas

# ============================================================
# STEP 1: Download Mistral-7B-Instruct-v0.2 (GGUF Quantized)
# ============================================================
import os

model_path = "/content/mistral-7b-instruct-v0.2.Q4_K_M.gguf"

if not os.path.exists(model_path):
    print("Downloading Mistral 7B GGUF model...")
    !wget -c https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF/resolve/main/mistral-7b-instruct-v0.2.Q4_K_M.gguf -O {model_path}
    print("Download completed successfully.")
else:
    print(f"Model already exists at: {model_path}")

# ============================================================
# STEP 3: Initialize Mistral 7B Engine
# ============================================================
from llama_cpp import Llama

# Load model and offload all layers to GPU VRAM for fast zero-shot inference
llm = Llama(
    model_path=model_path,
    n_gpu_layers=-1,      # Offload all 33 layers to GPU VRAM
    n_ctx=4096,           # Context window
    temperature=0.0,      # Deterministic classification output
    verbose=False
)

def run_mistral(prompt: str, max_tokens: int = 120) -> str:
    """Helper function to format prompts with Mistral instruction tags."""
    formatted_prompt = f"[INST] {prompt.strip()} [/INST]"
    output = llm(
        formatted_prompt,
        max_tokens=max_tokens,
        stop=["</s>", "\n\n\n"],
        echo=False
    )
    return output["choices"][0]["text"].strip()

print("Mistral 7B loaded and ready.")

Model already exists at: /content/mistral-7b-instruct-v0.2.Q4_K_M.gguf
Mistral 7B loaded and ready.


In [ ]:
# ============================================================
# STEP 5: Classification and Boundary Detection Function
# ============================================================
import json
import re

ALLOWED_DOC_TYPES = [
    "Cover Letter",
    "Certificate of Quality",
    "Packaging Specification",
    "BSE/TSE Declaration",
    "Material Description",
    "Supplier Qualification",
    "Chain of Custody",
    "Other",
    "Product Specification"
]

def analyze_page_boundary_and_type(page_idx: int, curr_text: str, prev_text: str = None, current_doc_type: str = None) -> dict:
    """
    Evaluates whether a page starts a new document or continues the previous one,
    and assigns the standardized document type.
    """
    if page_idx == 0:
        prompt = f"""You are a pharmaceutical document classifier.
Classify the following text into EXACTLY ONE category from this list:
- Cover Letter
- Certificate of Quality
- Packaging Specification
- BSE/TSE Declaration
- Material Description
- Supplier Qualification
- Chain of Custody
- Product Specification
- Other

Page Content:
\"\"\"{curr_text[:1200]}\"\"\"

Respond ONLY with a valid JSON object in this exact format:
{{"is_new_doc": "Yes", "doc_type": "<Doc Type>"}}"""
    else:
        prompt = f"""You are analyzing consecutive pages from a bundled pharmaceutical PDF.
Determine if the Current Page starts a NEW document or CONTINUES the previous document.

A page starts a NEW document ("is_new_doc": "Yes") if:
- It contains a new document title or header (e.g., "Certificate of Quality", "Packaging Specification").
- It contains a different lot number, revision number, or new product name.
- It is a standalone certificate/declaration.

A page CONTINUES the previous document ("is_new_doc": "No") if:
- It explicitly states "continued", "page 2 of 2", or has matching document control numbers.
- It is a continuation of tables/sections from the previous page.

Previous Document Type: {current_doc_type}
Previous Page Excerpt:
\"\"\"{prev_text[:600]}\"\"\"

Current Page Content:
\"\"\"{curr_text[:1200]}\"\"\"

If it is a new document, choose doc_type from:
["Cover Letter", "Certificate of Quality", "Packaging Specification", "BSE/TSE Declaration", "Material Description", "Supplier Qualification", "Chain of Custody", "Other"].
If it is NOT a new document, keep the previous doc_type: "{current_doc_type}".

Check all the page and find the Issue Date of this document Valid Date for this document , the date format should be all standardized to YYYY-MM-DD. If the date is not avaliable ot not found, enter default 0000-00-00


Respond ONLY with a valid JSON object in this exact format:
{{"is_new_doc": "Yes" or "No", "doc_type": "<Doc Type>"}}"""

    raw_response = run_mistral(prompt, max_tokens=80)

    # Fallback and JSON parsing handling
    try:
        json_match = re.search(r"\{.*?\}", raw_response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group(0))
            is_new = parsed.get("is_new_doc", "Yes").strip().capitalize()
            doc_type = parsed.get("doc_type", "Other").strip()
        else:
            raise ValueError("No JSON found in response")
    except Exception:
        is_new = "Yes" if "yes" in raw_response.lower() else "No"
        doc_type = current_doc_type if is_new == "No" and current_doc_type else "Other"

    # Match against standardized allowed types
    matched_type = "Other"
    for allowed in ALLOWED_DOC_TYPES:
        if allowed.lower() in doc_type.lower():
            matched_type = allowed
            break

    return {
        "is_new_doc": "Yes" if is_new.startswith("Y") else "No",
        "doc_type": matched_type
    }

In [ ]:
# ============================================================
# STEP 4: Build doc_pages from the PDF
# ============================================================
import fitz

def build_doc_pages(pdf_path):
    """
    Extract per-page text into the structure analyze_page_boundary_and_type()
    and the STEP 6 loop expect: a list of {"text": ..., "page_number": ...}.
    """
    doc = fitz.open(pdf_path)
    pages = [
        {"text": page.get_text(), "page_number": i + 1}
        for i, page in enumerate(doc)
    ]
    doc.close()
    return pages

doc_pages = build_doc_pages(pdf_path)
print(f"Built doc_pages: {len(doc_pages)} pages")
print("Sample page 0:", doc_pages[0]["text"][:200])

Built doc_pages: 39 pages
Sample page 0: 


In [ ]:
# ============================================================
# STEP 6: Execute Pipeline Loop and Track page_in_doc
# result is named as pipeline_results ( in dict) - with classsfication label
# Page 00 -> is_new_doc: Yes, doc_type: Product Specification, page_in_doc: 0
# ============================================================
pipeline_results = []
current_doc_type = None
page_in_doc_counter = 0

for i, page in enumerate(doc_pages):
    curr_text = page["text"]
    prev_text = doc_pages[i - 1]["text"] if i > 0 else None

    analysis = analyze_page_boundary_and_type(
        page_idx=i,
        curr_text=curr_text,
        prev_text=prev_text,
        current_doc_type=current_doc_type
    )

    # Manage page_in_doc index tracking
    if analysis["is_new_doc"] == "Yes":
        page_in_doc_counter = 0
        current_doc_type = analysis["doc_type"]
    else:
        page_in_doc_counter += 1
        analysis["doc_type"] = current_doc_type

    pipeline_results.append({
        "page": i,
        "is_new_doc": analysis["is_new_doc"],
        "doc_type": analysis["doc_type"],
        "page_in_doc": page_in_doc_counter
    })

    print(f"Page {i:02d} -> is_new_doc: {analysis['is_new_doc']}, doc_type: {analysis['doc_type']}, page_in_doc: {page_in_doc_counter}")

Page 00 -> is_new_doc: Yes, doc_type: Other, page_in_doc: 0
Page 01 -> is_new_doc: No, doc_type: Other, page_in_doc: 1
Page 02 -> is_new_doc: No, doc_type: Other, page_in_doc: 2
Page 03 -> is_new_doc: No, doc_type: Other, page_in_doc: 3
Page 04 -> is_new_doc: No, doc_type: Other, page_in_doc: 4
Page 05 -> is_new_doc: No, doc_type: Other, page_in_doc: 5
Page 06 -> is_new_doc: Yes, doc_type: Other, page_in_doc: 0
Page 07 -> is_new_doc: No, doc_type: Other, page_in_doc: 1
Page 08 -> is_new_doc: No, doc_type: Other, page_in_doc: 2
Page 09 -> is_new_doc: No, doc_type: Other, page_in_doc: 3
Page 10 -> is_new_doc: Yes, doc_type: Other, page_in_doc: 0


KeyboardInterrupt: 

In [ ]:
# ============================================================
# STEP 7: Eval: Format and Display Final Results
# ============================================================
import pandas as pd

# Display Pandas DataFrame
df_results = pd.DataFrame(pipeline_results)
print("=== Final Metadata DataFrame ===")
display(df_results)

# Display Structured JSON Output
print("\n=== Final JSON Metadata Output ===")
print(json.dumps(pipeline_results, indent=2))

=== Final Metadata DataFrame ===


,page,is_new_doc,doc_type,page_in_doc
0,0,Yes,Other,0
1,1,No,Other,1
2,2,No,Other,2
3,3,No,Other,3
4,4,No,Other,4
5,5,No,Other,5
6,6,Yes,Other,0
7,7,No,Other,1
8,8,No,Other,2
9,9,No,Other,3



=== Final JSON Metadata Output ===
[
  {
    "page": 0,
    "is_new_doc": "Yes",
    "doc_type": "Other",
    "page_in_doc": 0
  },
  {
    "page": 1,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 1
  },
  {
    "page": 2,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 2
  },
  {
    "page": 3,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 3
  },
  {
    "page": 4,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 4
  },
  {
    "page": 5,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 5
  },
  {
    "page": 6,
    "is_new_doc": "Yes",
    "doc_type": "Other",
    "page_in_doc": 0
  },
  {
    "page": 7,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 1
  },
  {
    "page": 8,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 2
  },
  {
    "page": 9,
    "is_new_doc": "No",
    "doc_type": "Other",
    "page_in_doc": 3
  },
  {
    "page": 10,


In [ ]:

"""
	Create one Document per page/section
   (each carrying its own metadata dict), then run SentenceSplitter on that list
"""
def chunk_document(pages, chunk_size=512, overlap=100, source="document"):
    """
    pages: list of dicts like {"text": "...", "page_number": 1, "section": "Specifications"}
    """
    splitter = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=overlap)

    documents = [
        Document(
            text=p["text"],
            metadata={"source": source, "page": p["page_number"], "section": p.get("section")}
        )
        for p in pages
    ]
    text_nodes = splitter.get_nodes_from_documents(documents)

    return [
        {"chunk_id": i, "source": source, "text": n.get_content(), **n.metadata}
        for i, n in enumerate(text_nodes)
    ]

In [ ]:
# ---- TEST: confirm chunk count, overlap, and structure look right ----
pages = [{"text": cleaned_text, "page_number": 1, "section": None}]
nodes = chunk_document(pages, chunk_size=512, overlap=100, source=pdf_path)

print(f"Created {len(nodes)} chunks")
print("-" * 60)
print("Sample node[0]:")
print(nodes[0])
print("-" * 60)
if len(nodes) > 1:
    print("End of chunk 0:", nodes[0]["text"][-100:])
    print("Start of chunk 1:", nodes[1]["text"][:100])
    print("(the overlapping words above should roughly match — that's chunk_overlap=100 at work)")

print(nodes[1])

Created 41 chunks
------------------------------------------------------------
Sample node[0]:
{'chunk_id': 0, 'source': 'Sample Documentation.pdf', 'text': 'BASF\nWe create\nchemistry\nRegXcellencef\nKollidone SR\nPRD-No:: 30071321\nProduct Specification\nWorkflow No.\nDAWF-2024-0299\nRevision 8\nEffective from: 01 Jul 2024\nStatus: final\nTest parameter\nRequirements\nTest method\nAppearance\nwhite to slightly yellowish,\nvisual\nfree flowing powder\nIdentification\nIdentification (IR)\nMust comply\nPM/00511\nIdentification (sulfate)\nMust comply\nPh.Eur. ("sodium lauril\nsulfate", ID test C)\nPurity tests\nContent of Povidone\n18.0 to 21.0 g/100g\n%\nPM/02047\n(calculated: Nitrogen\n0.126)\nContent of Polyvinylacetate\n76.0 to 82.0 g/100g\n%\nPMOO529QC\n(calculated: Saponification value x 0.1534)\nContent of sodium lauril sulfate\n<0.7 g/100g\n%)\nPMO2385QC\nContent of Silica (Si02)\n0.4 to 0.7 g/100g\n%)\nPM/02049\n(calculated: Si x 2.142)\nSulphated ash\nResidue on ignition\n<1.7\

#Part 3: Embeddings

In [ ]:
# ============================================
# embed_chunks()
# ============================================
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

def embed_chunks(nodes, embed_model=embed_model):
    """
    Compute an embedding vector for every chunk's text.

    Returns
    -------
    list[dict]
        Same nodes, each with an added 'embedding' key (list[float]).
    """
    texts = [n["text"] for n in nodes]
    vectors = embed_model.get_text_embedding_batch(texts, show_progress=True)

    for node, vector in zip(nodes, vectors):
        node["embedding"] = vector

    return nodes

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# ---- TEST: confirm every chunk got a same-length vector ----
embedded_nodes = embed_chunks(nodes)

dims = {len(n["embedding"]) for n in embedded_nodes}
print(f"Embedded {len(embedded_nodes)} chunks")
print(f"Embedding dimension(s) seen: {dims}")
print("First 5 values of chunk 0's embedding:", embedded_nodes[0]["embedding"][:5])
assert len(dims) == 1, "All chunks should share the same embedding dimension."
print("Block 3 passed: embeddings are populated and consistent.")

Generating embeddings:   0%|          | 0/41 [00:00<?, ?it/s]

Embedded 41 chunks
Embedding dimension(s) seen: {384}
First 5 values of chunk 0's embedding: [-0.10276257246732712, -0.07150886207818985, -0.0039042846765369177, -0.07925964891910553, 0.0729619488120079]
Block 3 passed: embeddings are populated and consistent.


## Store and Retrive Chunk with llamadex

store chunks into VectorStoreIndex object

In [ ]:
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex

# Reuse chunks + embeddings already computed in Cell 25/26 — no re-embedding
text_nodes = [
    TextNode(
        text=n["text"],
        metadata={k: v for k, v in n.items() if k not in ("text", "embedding")},
        embedding=n["embedding"],
    )
    for n in embedded_nodes
]

index = VectorStoreIndex(text_nodes, embed_model=embed_model)

# ---- TEST: confirm retrieval works through the real index ----
retriever = index.as_retriever(similarity_top_k=3)
test_query = "What are the storage condition?"
retrieved_nodes = retriever.retrieve(test_query)

print(f"Query: {test_query}\n")
for rank, n in enumerate(retrieved_nodes, start=1):
    print(f"#{rank} | similarity={n.score:.3f} | page={n.metadata.get('page')}")
    print(n.text.replace(chr(10), ' '))
    print("-" * 60)

Query: What are the storage condition?

#1 | similarity=0.364 | page=1
Conditions for safe storage; including any incompatibilities No applicable information available  Further information on storage conditions: Keep container tightly closed and dry. 8. Exposure Controls/Personal Protection No substance specific occupational exposure limits known. Advice on system design: Provide local exhaust ventilation to control dustslmists It is recommended that all dust control equipment such as local exhaust ventilation and material transport systems involved in handling of this product contain explosion relief vents or an explosion suppression system or an oxygen deficient environment; Ensure that dust-handling systems (such as exhaust ducts, dust collectors, vessels, and processing equipment) are designed in a manner to prevent the escape of dust into the work area (i.e-, there is no leakage from the equipment): Use only appropriately classified electrical equipment and powered industrial truc

In [ ]:
# ============================================
# compute_confidence()
# ============================================
def compute_confidence(retrieved_nodes):
    """
    Turn the retriever's similarity scores for a query into one confidence
    score for the answer that's about to be built from these chunks.

    A high average similarity across the top-k retrieved chunks means the
    model had strongly relevant context to work from; a low one means it's
    likely stretching or guessing.

    Returns
    -------
    dict: {"score": float (0-100), "label": "High" | "Medium" | "Low"}
    """
    if not retrieved_nodes:
        return {"score": 0.0, "label": "Low"}

    scores = [n.score for n in retrieved_nodes if n.score is not None]
    if not scores:
        return {"score": 0.0, "label": "Low"}

    avg_score = sum(scores) / len(scores)
    pct = round(max(0.0, min(avg_score, 1.0)) * 100, 1)

    if pct >= 70:
        label = "High"
    elif pct >= 45:
        label = "Medium"
    else:
        label = "Low"

    return {"score": pct, "label": label}


# ---- TEST: confirm confidence tracks retrieval quality ----
test_query = "What are the storage condition?"
retrieved_nodes = retriever.retrieve(test_query)
confidence = compute_confidence(retrieved_nodes)

print(f"Query: {test_query}")
print(f"Confidence: {confidence['score']}% ({confidence['label']})")

Query: What are the storage condition?
Confidence: 30.8% (Low)


# Part 4 Smart Prompting
Build clear prompt and inspect the model to cite source for its answer

## Pipeline

## build_context() vs build_prompt()

Two functions, two separate concerns: what data goes in, vs. how it's phrased for the model.

| | `build_context()` | `build_prompt()` |
|---|---|---|
| **Input** | Retrieved chunks (raw text + metadata) | The context string from `build_context()` + the user's question |
| **Job** | Turn scattered chunks into one clean, labeled block of text | Wrap that block in instructions the model actually follows |
| **Output** | `[Source: page 3, Certificate of Quality]\n"Store between 2-8°C..."` | Full prompt: role + rules + context block + question + citation instruction |
| **Changes if...** | Metadata shown changes (page vs. doc_type), or chunk separation format changes | Model behavior changes — stricter grounding, different citation style, different tone |

**Why split them:**
- Separation of concerns — swapping LLMs (e.g. different prompt format needs) only touches `build_prompt()`
- Easier debugging — check context labeling separately from instruction-following issues
- Reusability — same context can feed different prompt styles

**How they chain together:**
```python
context = build_context(retrieved_chunks)      # → "[Source: page 3]...\n[Source: page 5]..."
prompt  = build_prompt(context, user_question)  # → full instruction text sent to the LLM
answer  = run_mistral(prompt)
```

In [ ]:
# ============================================
# build_context()
# ============================================
def build_context(retrieved_nodes, max_chars=3000):
    """
    Turn retrieved chunks (raw text + metadata) into ONE labeled text block,
    ready to drop into a prompt.

    Each chunk gets a "[Source: page X, <section>]" tag directly above its
    text, so the model has something concrete to cite back later. We stop
    adding chunks once max_chars is hit, so we don't blow the model's
    context window.

    Parameters
    ----------
    retrieved_nodes : list
        Nodes returned by index.as_retriever().retrieve(query) — each has
        .text and .metadata (page, section, source).
    max_chars : int
        Soft cap on total context length.

    Returns
    -------
    str
        e.g. "[Source: page 3, Certificate of Quality]\n\"Store between 2-8C...\""
    """
    context_parts = []
    total_chars = 0

    for node in retrieved_nodes:
        page = node.metadata.get("page", "unknown")
        section = node.metadata.get("section") or "unspecified section"
        text = node.text.strip()

        label = f"[Source: page {page}, {section}]"
        block = f"{label}\n{text}"

        if total_chars + len(block) > max_chars:
            break

        context_parts.append(block)
        total_chars += len(block)

    return "\n\n".join(context_parts)


In [ ]:
# ============================================
# build_prompt()
# ============================================
def build_prompt(context, question):
    """
    Wrap the context block in instructions the model actually follows.

    Three rules matter most for a grounded RAG answer:
    1. Answer ONLY from the given context (no outside knowledge).
    2. Cite the source label for every factual claim.
    3. Say so explicitly if the answer isn't in the context, instead of guessing.

    Parameters
    ----------
    context : str
        Output of build_context().
    question : str
        The user's question.

    Returns
    -------
    str
        The full prompt to send to the LLM.
    """
    return f"""You are a document QA assistant. Answer the question using ONLY the context below.

Rules:
1. Every factual claim must be followed by its source citation, in the format (Source: page X, [Title of Document]).
2. If the context does not contain the answer, respond exactly: "I could not find this in the provided documents." Do not guess or use outside knowledge.
3. Keep the answer concise and directly address the question.

Context:
{context}

Question: {question}

Answer (with citations):"""


In [ ]:
# ---- TEST: chain build_context() + build_prompt() on a real retrieval ----
# Reuses the `retriever` and `index` built in Part 3 (Cell 29)
test_query = "What are the storage conditions?"
retrieved_nodes = retriever.retrieve(test_query)

context = build_context(retrieved_nodes)
prompt = build_prompt(context, test_query)

print("=== CONTEXT ===")
print(context)
print("\n=== FULL PROMPT ===")
print(prompt)

assert "[Source: page" in context, "Context is missing source labels — citations won't be possible."
assert "Rules:" in prompt and context in prompt, "Prompt didn't correctly wrap the context."
print("\nBlock 4 passed: context is labeled and prompt enforces citation + grounding rules.")

=== CONTEXT ===
[Source: page 1, unspecified section]
Conditions for safe storage; including any incompatibilities
No applicable information available 
Further information on storage conditions: Keep container tightly closed and dry.
8. Exposure Controls/Personal Protection
No substance specific occupational exposure limits known.
Advice on system design:
Provide local exhaust ventilation to control dustslmists
It is recommended that all dust control equipment such as local exhaust ventilation and material
transport systems involved in handling of this product contain explosion relief vents or an explosion
suppression system or an oxygen deficient environment;
Ensure that dust-handling systems (such as
exhaust ducts, dust collectors, vessels, and processing equipment) are designed in a manner to
prevent the escape of dust into the work area (i.e-, there is no leakage from the equipment): Use only
appropriately classified electrical equipment and powered industrial trucks_
Personal prot

# Part 5 Open-Source Model ( Mistral 7B (GGUF) )

Quality of result are similar with Gemini under a low cost


In [ ]:
"""
Connect Part 3's retriever + Part 4's prompt builders to the Mistral model
loaded earlier (before Part 2)
"""
Settings.embed_model = embed_model  # reuse Part 3's HuggingFaceEmbedding (all-MiniLM-L6-v2)
retriever = index.as_retriever(similarity_top_k=3)  # 'index' from Part 3

def answer_question(question, top_k=3):
    """The Part 4 -> Part 5 bridge: retrieve -> build_context -> build_prompt -> Mistral."""
    retrieved_nodes = retriever.retrieve(question)
    confidence = compute_confidence(retrieved_nodes)   # NEW
    context = build_context(retrieved_nodes)
    prompt = build_prompt(context, question)
    answer = run_mistral(prompt)
    return answer, confidence   # NEW: was just `return run_mistral(prompt)`


In [ ]:
# # ---- END-TO-END TEST: Part 1 -> Part 5 ----
# # test questions through the FULL pipeline and print each Q/A.

# test_queries = [
#     "What are the storage conditions for this product?",
#     "What sterilization method was used for this product?",
#     "Summarize the key specifications in this document.",
# ]

# pipeline_test_results = {}
# for q in test_queries:
# #     # change this line inside the test loop:
#       answer, confidence = answer_question(q)          # was: answer = answer_question(q)
#       pipeline_test_results[q] = answer
#       print(f"Q: {q}\nA: {answer}\nConfidence: {confidence['score']}% ({confidence['label']})\n{'-'*60}")

# # assert all(pipeline_test_results.values()), "One or more test queries returned an empty answer."
# print("\nEnd-to-end test passed: Part 1 (extraction) -> Part 2 (chunking) -> "
#       "Part 3 (embeddings/retrieval) -> Part 4 (build_context/build_prompt) -> "
#       "Part 5 (Mistral) produced answers for every test query.")


#Part 7: User interface

The following gradio UI has the main features. that are covered in the instructions

1. Upload documents
2. View Chat History ( export in .txt form ). DONE
3. Answer and source with confidence score


Minor Feature for better UI
1. Allow multiple folder uploads
2. loading indicator while processing

Upload Folder
        │
        ▼
Process Documents
(OCR + Chunking + Embeddings + Index)
        │
        ▼
Vector Store Ready
        │
        ▼
User asks question
        │
        ▼
Gradio
        │
        ▼
answer_question(question)
        │
        ▼
Retriever
        │
        ▼
Context Builder
        │
        ▼
Prompt Builder
        │
        ▼
Mistral 7B
        │
        ▼
Answer
        │
        ▼
Display in Chatbot

In [ ]:
# Run this in a Colab notebook or terminal
!pip install -q --upgrade "gradio>=4.44"

# 💡 Tip: Use a virtual environment for local installs if you're working locally


In [ ]:
import gradio as gr
import os
from llama_index.core.schema import TextNode
from llama_index.core import VectorStoreIndex

# ---------------------------------------------------------------------------
# Upload -> full pipeline (Part 1 -> Part 2 -> Part 3), rebuilds global index/retriever
# ---------------------------------------------------------------------------
def _file_path(f):
    """Gradio may hand back objects with .name or plain path strings depending on version."""
    return f.name if hasattr(f, "name") else f

def process_and_index(files):
    global index, retriever

    if not files:
        yield "⚠️ No files uploaded yet."
        return

    yield "⏳ Processing documents — extracting text, chunking, embedding..."

    all_nodes = []
    summaries = []
    try:
        for f in files:
            path = _file_path(f)
            filename = os.path.basename(path)

            raw_text = process_document(path)      # Part 1: PyMuPDF -> EasyOCR fallback
            cleaned = clean_text(raw_text)          # Part 1: cleanup

            pages = [{"text": cleaned, "page_number": 1, "section": None}]
            nodes = chunk_document(
                pages, chunk_size=512, overlap=100, source=filename
            )                                       # Part 2: chunking
            all_nodes.extend(nodes)
            summaries.append(f"📄 {filename}: {len(nodes)} chunk(s)")

        if not all_nodes:
            yield "⚠️ No usable text could be extracted from the uploaded file(s)."
            return

        embedded_nodes = embed_chunks(all_nodes)     # Part 3: embeddings

        text_nodes = [
            TextNode(
                text=n["text"],
                metadata={k: v for k, v in n.items() if k not in ("text", "embedding")},
                embedding=n["embedding"],
            )
            for n in embedded_nodes
        ]
        index = VectorStoreIndex(text_nodes, embed_model=embed_model)
        retriever = index.as_retriever(similarity_top_k=3)

        status = "✅ Processed " + str(len(files)) + " file(s):\n" + "\n".join(summaries)
        yield status

    except Exception as e:
        print(f"[process_and_index] failed: {e}")
        yield f"⚠️ Processing failed: {e}\nPlease check the file(s) and try again."


# ---------------------------------------------------------------------------
# Chat handling -> Part 5 (answer_question, which itself chains retriever ->
# build_context -> build_prompt -> Mistral). No changes made to answer_question().
# ---------------------------------------------------------------------------
def handle_chat(message, history):
    if not message or not message.strip():
        yield history, gr.update(value=""), gr.update(interactive=True)
        return

    if "retriever" not in globals() or retriever is None:
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": "⚠️ Please upload and process your documents first."},
        ]
        yield history, gr.update(value=""), gr.update(interactive=True)
        return

    history = history + [
        {"role": "user", "content": message},
        {
            "role": "assistant",
            "content": "Generating answer...",
            "metadata": {"status": "pending", "title": "🤔 Thinking"},
        },
    ]
    yield history, gr.update(value="", interactive=False), gr.update(interactive=False)

    try:
        answer, confidence = answer_question(message)
        if not answer or not str(answer).strip():
            raise ValueError("Empty response from model")

        emoji = {"High": "🟢", "Medium": "🟡", "Low": "🔴"}.get(confidence["label"], "⚪")
        content = (
            f"{answer}\n\n"
            f"---\n{emoji} **Confidence: {confidence['score']}% ({confidence['label']})**"
        )
        history[-1] = {"role": "assistant", "content": content}

    except Exception as e:
        print(f"[handle_chat] answer_question failed: {e}")
        history[-1] = {
            "role": "assistant",
            "content": (
                "⚠️ Sorry — that message wasn't sent successfully and no answer "
                "could be generated. Please try again in a moment."
            ),
        }

    yield history, gr.update(value=""), gr.update(interactive=True)
# ---------------------------------------------------------------------------
# Export chat history as .txt (unchanged from before)
# ---------------------------------------------------------------------------
def export_chat(history):
    import tempfile

    if not history:
        gr.Warning("There's no chat history yet to export.")
        return None

    lines = []
    for i in range(0, len(history) - 1, 2):
        user_turn = history[i]
        bot_turn = history[i + 1]
        lines.append(f"User: {user_turn.get('content', '')}")
        lines.append(f"Response: {bot_turn.get('content', '')}")

    content = "\n".join(lines)
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False, encoding="utf-8")
    tmp.write(content)
    tmp.close()
    return tmp.name


toggle_theme_js = """
() => {
    const shell = document.getElementById('app-shell');
    if (shell) { shell.classList.toggle('dark-mode'); }
}
"""

with gr.Blocks(title="RAG Document Assistant") as demo:
    with gr.Column(elem_id="app-shell"):
        with gr.Row():
            gr.Markdown("### 📚 RAG Document Assistant")
            theme_btn = gr.Button("🌙 / ☀️ Toggle Theme", scale=0)

        with gr.Row():
            with gr.Column(scale=2):
                chatbot = gr.Chatbot(label="Chat History", height=400)
                user_input = gr.Textbox(
                    placeholder="Ask a question about your documents...",
                    label="Your Question",
                )
                with gr.Row():
                    send_btn = gr.Button("📤 Send")
                    clear_btn = gr.Button("🗑️ Clear Chat")
                    export_btn = gr.DownloadButton("💾 Export Chat (.txt)")

            with gr.Column(scale=1):
                pdf_input = gr.File(
                    label="📄 Upload Document Folder",
                    file_types=[".pdf"],
                    file_count="directory",
                )
                process_btn = gr.Button("🔄 Re-process Documents")
                status_box = gr.Textbox(label="Status", interactive=False, lines=6)

        # Auto-process the moment files are uploaded
        pdf_input.upload(process_and_index, inputs=pdf_input, outputs=status_box)
        # Manual re-run still available
        process_btn.click(process_and_index, inputs=pdf_input, outputs=status_box)

        send_btn.click(
            handle_chat,
            inputs=[user_input, chatbot],
            outputs=[chatbot, user_input, send_btn],
        )
        user_input.submit(
            handle_chat,
            inputs=[user_input, chatbot],
            outputs=[chatbot, user_input, send_btn],
        )
        clear_btn.click(lambda: [], outputs=chatbot)

        theme_btn.click(None, None, None, js=toggle_theme_js)
        export_btn.click(export_chat, inputs=chatbot, outputs=export_btn)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c48c76a6cbef6410b3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Generating embeddings:   0%|          | 0/61 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c48c76a6cbef6410b3.gradio.live
